# Filter Google Local Washington data to Seattle

Find Seattle businesses in the metadata, keep only their reviews, and attach the business location to each review.

In [5]:
import pandas as pd

## 1. Seattle restaurants
Filter based on `, Seattle, WA <zip>` suffix. Filtering for restaurants based on if any of the categories contains "restaurant" (ex: `Restaurant`, `Chinese restaurant`, `Takeout Restaurant`).

In [9]:
meta = pd.read_json("data/meta-Washington.json.gz", lines=True)
in_seattle = meta["address"].str.contains(r", Seattle, WA \d{5}", na=False)
is_restaurant = meta["category"].apply(lambda cats: isinstance(cats, list) and any("restaurant" in c.lower() for c in cats))
seattle_meta = meta[in_seattle & is_restaurant].drop_duplicates("gmap_id")
seattle_ids = set(seattle_meta["gmap_id"])

seattle_meta.to_json("data/meta-Seattle.json.gz", orient="records", lines=True)
print(f"{len(seattle_ids):,} Seattle restaurants out of {meta['gmap_id'].nunique():,} businesses")
seattle_meta.head()

3,217 Seattle restaurants out of 120,642 businesses


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url
129,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",0x5490143a1a6be1a3:0xcdfc8b297798e8f2,None,47.678611,-122.355000,[Hamburger restaurant],4.0,14,None,"[[Wednesday, 2PM–2AM], [Thursday, 2PM–2AM], [F...","{'Service options': ['Delivery'], 'Highlights'...",Permanently closed,"[0x5490143b3a099223:0xfe6baadd7d3f8ff2, 0x5490...",https://www.google.com/maps/place//data=!4m2!3...
374,Baguette Vietnamese Sandwiches,"Baguette Vietnamese Sandwiches, 1509 E Madison...",0x54906ad1f83f2271:0xace5d4d692669216,None,47.614497,-122.312126,"[Restaurant, Sandwich shop, Vietnamese restaur...",5.0,1,None,"[[Wednesday, 11AM–3PM], [Thursday, 11AM–3PM], ...","{'Service options': ['Delivery'], 'Accessibili...",Permanently closed,"[0x5490148ada4ba629:0x75027ce1c1eb092c, 0x5490...",https://www.google.com/maps/place//data=!4m2!3...
429,Citizen Campfire Beer Garden,"Citizen Campfire Beer Garden, 706 Taylor Ave N...",0x549015dcda35c8b5:0x59acf0fabf61e550,None,47.625712,-122.345954,"[Bar & grill, Bar, Restaurant]",4.0,7,None,None,"{'Service options': ['Curbside pickup', 'No-co...",None,None,https://www.google.com/maps/place//data=!4m2!3...
506,Madeline Bread & Dessert,"Madeline Bread & Dessert, 1401 NE 42nd St, Sea...",0x549014f358189e23:0x7e6b6cb7dc73611,None,47.658208,-122.313014,[Restaurant],4.0,18,$$,"[[Tuesday, 10:30AM–9PM], [Wednesday, 10:30AM–9...","{'Service options': ['Takeout', 'Delivery'], '...",Permanently closed,"[0x549014f36a7d4685:0xaf653d60a168b032, 0x5490...",https://www.google.com/maps/place//data=!4m2!3...
637,Hansen Central,"Hansen Central, 1526 13th Ave S #1, Seattle, W...",0x54906b5052cf8c7d:0xf5182d591ee7748,None,47.589167,-122.315659,[Restaurant],5.0,2,None,None,{'Service options': ['Delivery']},None,None,https://www.google.com/maps/place//data=!4m2!3...


## 2. Seattle reviews

In [10]:
chunks = pd.read_json("data/review-Washington.json.gz", lines=True, chunksize=500_000, dtype={"user_id": str})
reviews = pd.concat(c[c["gmap_id"].isin(seattle_ids)] for c in chunks)
print(f"{len(reviews):,} Seattle reviews")

1,162,106 Seattle reviews


## 3. Attach business location to each review

In [11]:
business_cols = seattle_meta[["gmap_id", "name", "address", "latitude", "longitude", "category"]]
reviews = reviews.merge(business_cols, on="gmap_id", suffixes=("", "_business"))

reviews.to_json("data/review-Seattle.json.gz", orient="records", lines=True)
reviews.head()

,user_id,name,time,rating,text,pics,resp,gmap_id,name_business,address,latitude,longitude,category
0,114851985077823750062,Joshua McCormack,1476746560454,5.0,I feel like not a lot of people know about thi...,None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",47.678611,-122.355,[Hamburger restaurant]
1,110852791995453216926,Bettina Graupe,1388485302766,4.0,"If you want upscale or pretentious, look elsew...",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",47.678611,-122.355,[Hamburger restaurant]
2,105521314408384917549,J V,1413911526424,5.0,"It's a dive bar with great service, Do not exp...",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",47.678611,-122.355,[Hamburger restaurant]
3,113263257499923499987,Rachel Breitenbucher,1439148033917,5.0,Had a great experience last night visiting Ed'...,None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",47.678611,-122.355,[Hamburger restaurant]
4,112650185984208886011,Naomi Ishikawa,1325365056929,4.0,"Service is good. Food's alright, but...BUT......",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",47.678611,-122.355,[Hamburger restaurant]
